# 04h. 제출 11 구성 OOF 서비스 수치 (발표 21~26장 근거)

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: 발표 21~26장 숫자(1순위·상위 k개 정답률, 암별 F1, 헷갈린 쌍, 변이 수 구간, 확신도별 보류 기준, 대표 변이)를 같은 값으로 다시 계산
- 입력: 제출 11 구성(가중 XGB E + LR E·위치 구간 + LightGBM g3, 1:1:1) 저장 OOF + train 라벨 (재학습 없음)
- 최종 구성이 바뀌면 `PRED_KEY` 한 줄만 바꿔 다시 실행
- 기록표에는 적지 않음 (결과는 이 노트북 출력), Run All 가능 (수 초)

## 0. 환경 설정 (수정하지 말고 그대로 실행)

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


## 1. 예측·라벨 불러오기

In [2]:
import json
import numpy as np
import pandas as pd

# 제출 11 구성 저장 OOF (최종 구성이 바뀌면 이 줄만 수정)
PRED_KEY = "A_model__ensemble_E_xgb_position_lr_lgbm_g3_equal_1110"
PRED = os.path.join(REPO_DIR, "experiments", "preds", PRED_KEY + "__")

train = pd.read_csv(os.path.join(REPO_DIR, "data", "train.csv"))
classes = json.load(open(PRED + "meta.json"))["classes"]
oof = np.load(PRED + "oof.npy")
assert len(oof) == len(train), "OOF 행 수와 train 행 수가 다름"

y = np.array([classes.index(c) for c in train["SUBCLASS"]])
pred = oof.argmax(1)
correct = pred == y
print("환자 수:", len(y), "| 암종 수:", len(classes))

환자 수: 6201 | 암종 수: 26


## 2. Macro F1, 암별 정밀도·재현율·F1

In [3]:
rows = []
for k, c in enumerate(classes):
    tp = ((pred == k) & (y == k)).sum(); fp = ((pred == k) & (y != k)).sum(); fn = ((pred != k) & (y == k)).sum()
    rows.append({"암": c, "환자 수": (y == k).sum(), "예측 수": (pred == k).sum(),
                 "정밀도": tp / max(tp + fp, 1), "재현율": tp / (tp + fn), "F1": 2 * tp / max(2 * tp + fp + fn, 1)})
per_class = pd.DataFrame(rows).sort_values("F1", ascending=False)
print("Macro F1:", round(per_class["F1"].mean(), 4), "| 1순위 정확도:", round(correct.mean(), 4))
print(per_class.round(3).to_string(index=False))

Macro F1: 0.5284 | 1순위 정확도: 0.5167
     암  환자 수  예측 수   정밀도   재현율    F1
  SKCM   276   238 0.966 0.833 0.895
   ACC    72    56 1.000 0.778 0.875
  COAD   223   196 0.842 0.740 0.788
  THCA   324   249 0.807 0.620 0.702
  UCEC   198   161 0.776 0.631 0.696
  TGCT   124   117 0.667 0.629 0.647
  LUSC   178   171 0.608 0.584 0.596
   LGG   229   382 0.474 0.790 0.592
  STES   379   394 0.579 0.602 0.590
  BLCA   104    80 0.675 0.519 0.587
  LAML   158   125 0.656 0.519 0.580
  BRCA   786   734 0.565 0.528 0.546
  DLBC    38    13 1.000 0.342 0.510
  LIHC   158   107 0.589 0.399 0.475
  LUAD   184   125 0.584 0.397 0.472
    OV   253   409 0.367 0.593 0.453
  KIRC   334   347 0.429 0.446 0.438
  CESC   155   152 0.434 0.426 0.430
GBMLGG   461   300 0.533 0.347 0.420
  HNSC   223   189 0.455 0.386 0.417
  PAAD   120   100 0.450 0.375 0.409
  PRAD   266   491 0.305 0.564 0.396
 KIPAN   515   504 0.379 0.371 0.375
  PCPG   147   198 0.308 0.415 0.354
  THYM    98   181 0.243 0.449 0.315
  S

## 3. 상위 k개 정답 포함률

In [4]:
order = np.argsort(-oof, 1)
for k in (1, 2, 3, 5):
    print(f"상위 {k}개 정답 포함률:", round(np.mean([y[i] in order[i, :k] for i in range(len(y))]), 4))

상위 1개 정답 포함률: 0.5167
상위 2개 정답 포함률: 0.6925
상위 3개 정답 포함률: 0.7731
상위 5개 정답 포함률: 0.86


## 4. 가장 많이 헷갈린 쌍 (실제 → 예측)

In [5]:
cm = pd.crosstab(train["SUBCLASS"], np.array(classes)[pred])
pairs = sorted(((a, b, cm.loc[a, b]) for a in cm.index for b in cm.columns if a != b), key=lambda t: -t[2])[:8]
print("헷갈린 쌍:", pairs)

헷갈린 쌍: [('GBMLGG', 'LGG', np.int64(189)), ('KIPAN', 'KIRC', np.int64(157)), ('KIRC', 'KIPAN', np.int64(96)), ('BRCA', 'OV', np.int64(88)), ('BRCA', 'PRAD', np.int64(76)), ('KIPAN', 'PRAD', np.int64(39)), ('SARC', 'PRAD', np.int64(38)), ('THCA', 'PRAD', np.int64(37))]


## 5. 변이 유전자 수 구간별 정확도

In [6]:
genes = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
n_mut = ((train[genes].astype(str) != "WT") & train[genes].notna()).sum(axis=1).values
for lo, hi in [(0, 0), (1, 2), (3, 5), (6, 10), (11, 20), (21, 50), (51, 10**6)]:
    m = (n_mut >= lo) & (n_mut <= hi)
    print(f"변이 {lo}~{hi}개: {m.sum()}명, 정확도 {correct[m].mean():.3f}")

변이 0~0개: 94명, 정확도 0.319
변이 1~2개: 370명, 정확도 0.424
변이 3~5개: 762명, 정확도 0.480
변이 6~10개: 1262명, 정확도 0.437
변이 11~20개: 1627명, 정확도 0.476
변이 21~50개: 1253명, 정확도 0.555
변이 51~1000000개: 833명, 정확도 0.755


## 6. 확신도별 판단 보류 기준표·구간별 정답률

In [7]:
conf = oof.max(1)
for t in (0.3, 0.4, 0.5, 0.6, 0.7):
    s = conf >= t
    print(f"기준 {t}: 바로 제시 {s.mean():.3f} (정확도 {correct[s].mean():.3f}) / 보류 정확도 {correct[~s].mean():.3f}")
for lo, hi in [(0, .2), (.2, .4), (.4, .6), (.6, .8), (.8, 1.01)]:
    m = (conf >= lo) & (conf < hi)
    print(f"확신도 {lo}~{hi}: {m.sum()}명, 평균 확신도 {conf[m].mean():.3f}, 실제 정답률 {correct[m].mean():.3f}")

기준 0.3: 바로 제시 0.637 (정확도 0.670) / 보류 정확도 0.247
기준 0.4: 바로 제시 0.505 (정확도 0.730) / 보류 정확도 0.299
기준 0.5: 바로 제시 0.385 (정확도 0.794) / 보류 정확도 0.343
기준 0.6: 바로 제시 0.288 (정확도 0.849) / 보류 정확도 0.382
기준 0.7: 바로 제시 0.192 (정확도 0.927) / 보류 정확도 0.419
확신도 0~0.2: 913명, 평균 확신도 0.168, 실제 정답률 0.189
확신도 0.2~0.4: 2155명, 평균 확신도 0.283, 실제 정답률 0.346
확신도 0.4~0.6: 1348명, 평균 확신도 0.495, 실제 정답률 0.571
확신도 0.6~0.8: 982명, 평균 확신도 0.687, 실제 정답률 0.745
확신도 0.8~1.01: 803명, 평균 확신도 0.908, 실제 정답률 0.976


## 7. 대표 변이 보유 환자의 1순위 정확도

In [8]:
G = train[genes].astype(str)
for name, gene, pat in [("BRAF V600E", "BRAF", "V600E"), ("PIK3CA H1047R", "PIK3CA", "H1047R"), ("NPM1", "NPM1", None),
                        ("APC", "APC", None), ("IDH1 R132", "IDH1", "R132"), ("VHL", "VHL", None)]:
    m = (G[gene] != "WT") & (G[gene] != "nan")
    if pat: m &= G[gene].str.contains(pat)
    print(f"{name}: {m.sum()}명, 1순위 정확도 {correct[m.values].mean():.3f}")


BRAF V600E: 321명, 1순위 정확도 0.928
PIK3CA H1047R: 147명, 1순위 정확도 0.925
NPM1: 56명, 1순위 정확도 0.946
APC: 367명, 1순위 정확도 0.747
IDH1 R132: 411명, 1순위 정확도 0.511
VHL: 326명, 1순위 정확도 0.497


## 8. 결론 (Mac 기준)

- Macro F1 0.5284, 1순위 정확도 0.5167, 상위 3개 0.7731, 상위 5개 0.860 → 발표 21~26장 수치와 일치
- 헷갈린 쌍 상위 3개: GBMLGG→LGG 189, KIPAN→KIRC 157, KIRC→KIPAN 96 (라벨이 겹치는 쌍둥이 암종)
- 확신도 0.5 이상만 바로 제시: 38.5% 환자, 정확도 0.794 / 0.7 이상: 19.2%, 정확도 0.927
- 대표 변이: BRAF V600E 321명 0.928, PIK3CA H1047R 147명 0.925, NPM1 56명 0.946
- 저장 OOF만 사용하므로 라이브러리 버전과 관계없이 같은 값